# ARIA — Entrenar un adaptador LoRA (gratis, en la nube)

Este cuaderno entrena un **adaptador LoRA** (pequeño, menos de 300 MB) para el modelo ARIA,
usando el archivo `aria-chat.jsonl` que genera el repositorio con:

```
npm run db:dataset
```

- **No usa tu computador**: corre en la GPU gratuita de Kaggle (o Colab).
- **No sube tus datos a ningún lado público**: el archivo se queda en tu sesión privada.
- **Formato del archivo**: una línea JSON por conversación: `{"messages":[{"role":...},...]}`.

> Aviso honesto: este cuaderno es un **punto de partida**. Las librerías (unsloth, trl, peft)
cambian de versión seguido. Si alguna celda falla por un nombre de parámetro, mirá las notas
en rojo de cada sección y los enlaces del final.

> **Si Kaggle se reinicia**: no se perdió el cuaderno, solo la sesión. El código está guardado
en el repo; los archivos de `/kaggle/working` sí se pierden. La sección **6** explica cómo
reanudar el entrenamiento sin empezar de cero.

## 0. Antes de empezar (2 minutos)

1. Barra lateral derecha → **Internet: activado**.
2. **Accelerator: GPU T4 x2** (o P100).
3. Subí `aria-chat.jsonl` como **dataset privado** y adjuntalo al cuaderno.
4. Ejecutá las celdas **de arriba hacia abajo**.
5. Al terminar: **Save Version → Save & Run All**. Eso conserva el `aria-lora.zip` en el
   *Output* aunque la sesión se caiga.

> La celda de instalación **no desinstala** el `torch` con CUDA que ya trae Kaggle:
> hacerlo deja el entorno a medias. Si Kaggle pide reiniciar el kernel, reiniciá y
> seguí desde la celda de **verificación de versiones** (no repitas la instalación).

## 1. Instalar las librerías

En Kaggle: activá **Internet** en la barra lateral derecha antes de ejecutar esta celda.

In [ ]:
# Instalación de la pila de entrenamiento (QLoRA) con Unsloth.
# Kaggle ya trae PyTorch con CUDA: NO lo desinstales ni lo reinstales a mano.
# Eso deja `torch` a medias (sin torch.cuda) y rompe peft, bitsandbytes y unsloth.
#
# Unsloth 2026.11.4 instala sus propias versiones compatibles de transformers y trl.
# Si más adelante esa versión no existe para la imagen de Kaggle, quitá los `==`.
!pip install -q --upgrade "unsloth==2026.11.4" "unsloth_zoo==2026.11.4"

In [ ]:
# Verificación: si algo falló en la instalación, se ve acá y no 20 minutos después.
# Unsloth debe importarse antes que transformers/trl para no perder optimizaciones.
import importlib, platform, sys

print("Python:", sys.version.split()[0], "|", platform.platform())

mods = {}
for nombre in ["unsloth", "torch", "transformers", "trl", "peft", "datasets", "bitsandbytes"]:
    try:
        mods[nombre] = importlib.import_module(nombre)
        print(f"  {nombre:14s} {getattr(mods[nombre], '__version__', 'ok')}")
    except Exception as e:
        mods[nombre] = None
        print(f"  {nombre:14s} FALTA -> {type(e).__name__}: {e}")

torch = mods["torch"]
if torch is None or not hasattr(torch, "cuda"):
    raise RuntimeError(
        "PyTorch quedó roto (no tiene torch.cuda). No desinstales torch en Kaggle: "
        "hacé Factory reset o abrí una sesión nueva y ejecutá de nuevo la instalación "
        "y esta verificación."
    )

version_cuda = getattr(getattr(torch, "version", None), "cuda", None)
print("CUDA build:", version_cuda)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError("Kaggle no tiene GPU: Settings -> Accelerator -> GPU T4 x2")

faltan = [n for n, m in mods.items() if m is None]
if faltan:
    raise RuntimeError(f"Faltan módulos: {', '.join(faltan)}. Volvé a ejecutar la instalación.")

## 2. Localizar y validar el archivo de datos

Tenés dos opciones:

1. **Recomendada**: subí `aria-chat.jsonl` como un *dataset* **privado** de Kaggle y adjuntalo a este cuaderno.
2. Arrastrá `aria-chat.jsonl` a la carpeta de trabajo (queda en `/kaggle/working/`).

Ajustá `RUTA_DATOS` abajo según dónde te haya quedado. La celda falla con un mensaje claro
si el archivo no está, en vez de romper más adelante.

**Recordá**: este archivo tiene tus conversaciones. Mantenelo privado.

In [ ]:
from pathlib import Path

# Busca cualquier .jsonl en TODO /kaggle/input y /kaggle/working.
# Así funciona aunque el dataset haya quedado en una subcarpeta o con otro nombre.
def jsonl_en(base):
    base = Path(base)
    return sorted(p for p in base.rglob("*.jsonl") if p.is_file()) if base.exists() else []


candidatos = jsonl_en("/kaggle/input") + jsonl_en("/kaggle/working")

print("Archivos .jsonl encontrados:")
for p in candidatos:
    print(f"  - {p}  ({p.stat().st_size / 1024:.1f} KB)")

if not candidatos:
    print("\nNo hay ningún .jsonl. Contenido de /kaggle/input:")
    for p in sorted(Path("/kaggle/input").rglob("*")):
        print("  ", p)
    raise FileNotFoundError(
        "No hay ningún .jsonl adjunto. En Kaggle: Add Input -> Datasets -> "
        "tu dataset privado -> Add. Después volvé a ejecutar esta celda."
    )

# Si hay varios, usa el más grande (el dataset suele ser el archivo grande).
RUTA_DATOS = max(candidatos, key=lambda p: p.stat().st_size)

print("\nUsando:", RUTA_DATOS)
print("Tamaño (KB):", round(RUTA_DATOS.stat().st_size / 1024, 1))

if RUTA_DATOS.name != "aria-chat.jsonl":
    print("AVISO: el archivo no se llama aria-chat.jsonl, pero el formato es el mismo.")


In [ ]:
# Validación del JSONL: evita entrenar con un archivo corrupto o vacío.
import json
from collections import Counter

ROLES_OK = {"system", "user", "assistant"}
filas, errores = [], []
roles = Counter()

with RUTA_DATOS.open("r", encoding="utf-8") as f:
    for n, linea in enumerate(f, start=1):
        linea = linea.strip()
        if not linea:
            continue
        try:
            obj = json.loads(linea)
        except json.JSONDecodeError as e:
            errores.append(f"línea {n}: JSON inválido ({e})")
            continue

        msgs = obj.get("messages")
        if not isinstance(msgs, list) or not msgs:
            errores.append(f"línea {n}: falta 'messages' o está vacío")
            continue
        if any(not isinstance(m, dict) or m.get("role") not in ROLES_OK for m in msgs):
            errores.append(f"línea {n}: rol inválido (solo system/user/assistant)")
            continue
        if not any(m["role"] == "assistant" for m in msgs):
            errores.append(f"línea {n}: no tiene ningún mensaje 'assistant'")
            continue

        for m in msgs:
            roles[m["role"]] += 1
        filas.append(obj)

print("Conversaciones válidas:", len(filas))
print("Errores:", len(errores))
for e in errores[:5]:
    print("  -", e)
print("Mensajes por rol:", dict(roles))

if not filas:
    raise ValueError("No hay ni una conversación válida. Revisá cómo se generó aria-chat.jsonl.")

if len(filas) < 20:
    print(f"\nAVISO: solo {len(filas)} conversaciones. Con tan pocos ejemplos el modelo MEMORIZA;")
    print("sirve para probar el circuito, no para tener un buen adaptador.")


In [ ]:
# Cargar, deduplicar y separar una parte para evaluar.
import json

from datasets import load_dataset

ds = load_dataset("json", data_files=str(RUTA_DATOS), split="train")
print("Ejemplos cargados:", len(ds))
print("Roles del primero:", [m["role"] for m in ds[0]["messages"]])

# Quitar conversaciones repetidas (mismo JSON exacto).
vistos, indices = set(), []
for i, ej in enumerate(ds):
    clave = json.dumps(ej["messages"], sort_keys=True, ensure_ascii=False)
    if clave not in vistos:
        vistos.add(clave)
        indices.append(i)

ds = ds.select(indices)
print("Después de deduplicar:", len(ds))

# Con pocos ejemplos no da para separar evaluación: entrenamos con todo.
if len(ds) >= 20:
    ds = ds.train_test_split(test_size=0.1, seed=42)
    print("Entrenamiento:", len(ds["train"]), "| Evaluación:", len(ds["test"]))
else:
    from datasets import DatasetDict
    ds = DatasetDict({"train": ds})
    print("Entrenamiento:", len(ds["train"]), "| Evaluación: (omitida, hay pocos ejemplos)")


In [ ]:
# Estimar el largo de las conversaciones para elegir MAX_LARGO sin truncar de más.
largos = []
for ej in ds["train"]:
    texto = " ".join(m.get("content", "") for m in ej["messages"])
    largos.append(len(texto) // 4)  # regla gruesa: ~4 caracteres por token

largos.sort()
def pct(p):
    return largos[min(len(largos) - 1, int(len(largos) * p))]

print("Tokens aproximados por conversación:")
print("  mediana:", pct(0.50), "| p90:", pct(0.90), "| máximo:", largos[-1])

MAX_LARGO = 1024
if pct(0.90) > MAX_LARGO:
    print(f"\nOJO: el 10% más largo supera {MAX_LARGO} tokens y se va a truncar.")
    print("Subí MAX_LARGO (1024 -> 2048) si tenés VRAM; en T4 conviene dejarlo en 1024.")
else:
    print(f"\nMAX_LARGO={MAX_LARGO} cubre ~el 90% de las conversaciones.")


## 3. Cargar el modelo base

**Importante (compatibilidad con Cloudflare):** el adaptador solo sirve sobre una base LoRA
de Workers AI, y la documentación exige que `adapter_config.json` tenga
`model_type` ∈ {`mistral`, `gemma`, `llama`}. Por eso **Qwen no sirve** para subirlo, aunque
aparezca en el catálogo. Las dos opciones realistas en GPU gratuita son:

| Opción | Base para entrenar (4 bits) | `base_model_name_or_path` al subir | Base Cloudflare | `model_type` | VRAM |
|---|---|---|---|---|---|
| `llama-3.2-3b` | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` | `meta-llama/Llama-3.2-3B-Instruct` | `@cf/meta/llama-3.2-3b-instruct` | `llama` | ~8 GB |
| `mistral-7b` | `unsloth/mistral-7b-instruct-v0.2-bnb-4bit` | `mistralai/Mistral-7B-Instruct-v0.2` | `@cf/mistral/mistral-7b-instruct-v0.2-lora` | `mistral` | ~14 GB |

**Ojo con la base:** Cloudflare valida `base_model_name_or_path` contra su lista de
bases permitidas. La ruta de Unsloth (`unsloth/...-bnb-4bit`) **no** sirve; hay que
sobrescribirla con la base canónica de la tabla. La celda de guardado ya lo hace.

Se carga en **4 bits** para que entre en una GPU de 16 GB. Si tu conjunto de datos es muy
chico (pocos ejemplos), el modelo va a *memorizar*: sirve para **probar que el circuito
funciona**, no para tener un buen modelo todavía.

In [ ]:
from unsloth import FastLanguageModel

# Elegí una de las dos opciones de la tabla de arriba.
ELEGIDO = "llama-3.2-3b"

MODELOS = {
    "llama-3.2-3b": {
        "hf": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "hf_base": "meta-llama/Llama-3.2-3B-Instruct",
        "cloudflare": "@cf/meta/llama-3.2-3b-instruct",
        "model_type": "llama",
    },
    "mistral-7b": {
        "hf": "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
        "hf_base": "mistralai/Mistral-7B-Instruct-v0.2",
        "cloudflare": "@cf/mistral/mistral-7b-instruct-v0.2-lora",
        "model_type": "mistral",
    },
}

INFO = MODELOS[ELEGIDO]
MODELO_BASE = INFO["hf"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELO_BASE,
    max_seq_length=MAX_LARGO,
    load_in_4bit=True,
)
print("Modelo cargado:", MODELO_BASE)
print("Se subirá como LoRA de:", INFO["cloudflare"], "| model_type:", INFO["model_type"])


## 4. Configurar el adaptador LoRA

Usamos **rango 8** (`r=8`) a propósito: las LoRA de Cloudflare aceptan rango **<= 8**
(hasta 32) y el archivo final queda bien por debajo de los 300 MB permitidos.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # Cloudflare exige rango <= 8 (o hasta 32 según la doc)
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Adaptador configurado (r=8).")


## 5. Entrenar

Convertimos cada conversación a texto con la *plantilla de chat* del modelo y entrenamos.

> Nota de versión: en algunas versiones de `trl`, `dataset_text_field` y `max_seq_length` van
> dentro de `SFTConfig`; en otras, como argumentos de `SFTTrainer`. La celda detecta la firma
> instalada y arma los argumentos sola. Si igual falla, el error dice qué parámetro sobra.

Los *checkpoints* se guardan al final de cada época en `salida/`: eso es lo que permite reanudar si
Kaggle reinicia la sesión.

In [ ]:
import inspect
import torch
from trl import SFTTrainer

try:
    from trl import SFTConfig
except ImportError:  # versiones muy viejas
    SFTConfig = None


def a_texto(ejemplo):
    return {
        "text": tokenizer.apply_chat_template(
            ejemplo["messages"], tokenize=False, add_generation_prompt=False
        )
    }


ds_texto = ds.map(a_texto)
train_ds = ds_texto["train"]
eval_ds = ds_texto["test"] if "test" in ds_texto else None

SALIDA = "salida"

bf16 = torch.cuda.is_bf16_supported()
comunes = dict(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    output_dir=SALIDA,
    report_to="none",
    seed=3407,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=not bf16,
    bf16=bf16,
    packing=False,
)

kwargs_trainer = dict(model=model, train_dataset=train_ds)
if eval_ds is not None:
    kwargs_trainer["eval_dataset"] = eval_ds

firma_trainer = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in firma_trainer:
    kwargs_trainer["processing_class"] = tokenizer
else:
    kwargs_trainer["tokenizer"] = tokenizer

if SFTConfig is not None:
    firma_cfg = inspect.signature(SFTConfig.__init__).parameters
    cfg = dict(comunes)
    if "dataset_text_field" in firma_cfg:
        cfg["dataset_text_field"] = "text"
    else:
        kwargs_trainer["dataset_text_field"] = "text"
    if "max_seq_length" in firma_cfg:
        cfg["max_seq_length"] = MAX_LARGO
    elif "max_length" in firma_cfg:
        cfg["max_length"] = MAX_LARGO
    else:
        kwargs_trainer["max_seq_length"] = MAX_LARGO
    kwargs_trainer["args"] = SFTConfig(**cfg)
else:
    kwargs_trainer.update(comunes)
    kwargs_trainer["dataset_text_field"] = "text"
    kwargs_trainer["max_seq_length"] = MAX_LARGO

entrenador = SFTTrainer(**kwargs_trainer)
print("Listo para entrenar | fp16:", not bf16, "| bf16:", bf16)


In [ ]:
# Reanuda solo si hay checkpoints de una sesión anterior (ver sección 6).
import glob

checkpoints = sorted(glob.glob(f"{SALIDA}/checkpoint-*"))
if checkpoints:
    print("Checkpoints encontrados, reanudo desde el último:", checkpoints[-1])
    entrenador.train(resume_from_checkpoint=True)
else:
    print("Sin checkpoints previos: entreno desde cero.")
    entrenador.train()


## 6. Probar el adaptador antes de descargarlo

Si el modelo responde algo coherente con el estilo del dataset, el circuito funciona. No hace
falta que sea perfecto: con pocos ejemplos va a memorizar.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

pregunta = "Hola ARIA, ¿quién sos y en qué podés ayudarme?"
mensajes = [{"role": "user", "content": pregunta}]

entrada = tokenizer.apply_chat_template(
    mensajes, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

salida = model.generate(
    input_ids=entrada,
    max_new_tokens=160,
    temperature=0.7,
    do_sample=True,
)
print(tokenizer.decode(salida[0][entrada.shape[-1]:], skip_special_tokens=True))


## 7. Guardar el adaptador compatible con Cloudflare

Se guarda en la carpeta `aria-lora/`, con el `adapter_config.json` y el
`adapter_model.safetensors` que pide Cloudflare. La celda **corrige y verifica** lo que
Cloudflare exige antes de comprimir:

- `model_type` presente y válido (`llama` o `mistral`).
- `base_model_name_or_path` con la base canónica que acepta Cloudflare (no la de Unsloth).
- `r` <= 8.
- Los dos archivos con el nombre exacto.
- Peso total < 300 MB.

In [ ]:
import json
import shutil
from pathlib import Path

CARPETA = Path("aria-lora")
CARPETA.mkdir(exist_ok=True)

model.save_pretrained(CARPETA)
tokenizer.save_pretrained(CARPETA)

# Cloudflare exige model_type y solo acepta mistral/gemma/llama.
# También valida la base: hay que usar la canónica, no la ruta de Unsloth.
cfg_path = CARPETA / "adapter_config.json"
cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
cfg["model_type"] = INFO["model_type"]
cfg["base_model_name_or_path"] = INFO["hf_base"]
cfg_path.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")

r = int(cfg.get("r", 0) or 0)
assert r <= 32, f"r={r} supera el máximo de Cloudflare (32)"
if r > 8:
    print(f"AVISO: r={r}; la doc recomienda <= 8")

pesos = CARPETA / "adapter_model.safetensors"
assert pesos.exists(), "Falta adapter_model.safetensors"
assert cfg_path.exists(), "Falta adapter_config.json"

mb = sum(p.stat().st_size for p in CARPETA.iterdir() if p.is_file()) / 1024**2
print("model_type:", cfg["model_type"], "| r:", r)
print("base:", cfg["base_model_name_or_path"], "->", INFO["cloudflare"])
print(f"Peso total: {mb:.1f} MB (límite 300 MB)")
assert mb < 300, "El adaptador supera los 300 MB: bajá r o target_modules"

print("Archivos:", sorted(p.name for p in CARPETA.iterdir()))


In [ ]:
# Comprimir para bajarlo desde el panel Output de Kaggle.
zip_path = Path(shutil.make_archive("aria-lora", "zip", root_dir=CARPETA))
print("Comprimido:", zip_path.resolve())
print("Tamaño (MB):", round(zip_path.stat().st_size / 1024**2, 1))
print("Bajalo desde el panel de archivos (Output) o con Save Version.")


## 8. Si Kaggle se reinicia (no perder el avance)

Kaggle borra `/kaggle/working` cuando la sesión se reinicia o expira. Lo que **sí** sobrevive:

1. **El cuaderno guardado** (el código). Versioná este `.ipynb` en el repo, como ya está.
2. **El Output de una versión guardada**: *Save Version → Save & Run All*. El `aria-lora.zip`
   queda descargable aunque la sesión muera.
3. **Los checkpoints** de `salida/`, solo mientras la sesión siga viva. Por eso se guardan
   al final de cada época.

Para reanudar después de un reinicio:

1. Volvé a ejecutar **instalación** (si el kernel es nuevo) y **verificación**.
2. Ejecutá **datos** (secciones 2) y **modelo + LoRA** (secciones 3 y 4) en el mismo orden.
3. Ejecutá la celda de **entrenamiento**: si encuentra `salida/checkpoint-*`, reanuda sola.
4. Si el reinicio borró `salida/`, no hay magia: se empieza de cero. Para eso está el punto 2.

> Truco: subí el `aria-lora.zip` como **dataset privado** y adjuntalo la próxima vez. Así el
> adaptador también sobrevive a cualquier reinicio.

## Siguientes pasos

1. **Bajá** `aria-lora.zip` desde el panel *Output* de Kaggle.
2. **No lo subas** a ningún repositorio público de Hugging Face (regla del proyecto).
3. Para **usarlo en ARIA**, subilo a Cloudflare (necesita token con *Workers AI: Edit*):

   ```bash
   npx wrangler ai finetune create "@cf/meta/llama-3.2-3b-instruct" aria-lora ./aria-lora
   npx wrangler ai finetune list
   ```

   Y en el Worker se invoca con `env.AI.run(modelo, { messages, lora: "<id-o-nombre>" })`.

4. **Ojo**: hoy la familia de bases `-lora` de Cloudflare devuelve **error 500**, incluso sin
   adaptador. El adaptador se guarda igual, para cuando eso se resuelva. La base
   `llama-3.2-3b-instruct` (sin sufijo) responde 200; falta confirmar que acepte el adaptador.
5. Cuando tengas **muchas más conversaciones**, volvé a generar el conjunto con
   `npm run db:dataset` y repetí este cuaderno. Con 6 ejemplos no hay modelo útil.

### Enlaces

- Unsloth: https://github.com/unslothai/unsloth
- TRL / SFTTrainer: https://huggingface.co/docs/trl
- LoRA en Cloudflare: https://developers.cloudflare.com/workers-ai/features/fine-tunes/loras/